In [1]:
import psi4

psi4.set_memory('1000mb')
psi4.core.set_num_threads(1)

zmatrix = '''\
H11
O11  H11  0.9600
C11  O11  1.4000  H11  108.0000
H12  C11  1.1000  O11  112.0000  H11  -60.0000
H13  C11  1.1000  O11  112.0000  H11   60.0000
H14  C11  1.1000  O11  111.8699  H11 -180.0000
'''

universe = psi4.geometry(zmatrix)
universe.update_geometry()
universe.print_out()

universe.save_xyz_file('test.xyz', False)


  Memory set to 953.674 MiB by Python driver.
  Threads set to 1 by Python driver.
    Molecular point group: cs
    Full point group: Cs

    Geometry (in Angstrom), charge = 0, multiplicity = 1:

       Center              X                  Y                   Z               Mass       
    ------------   -----------------  -----------------  -----------------  -----------------
         H           -0.661515257712     1.144760206605     0.000000000000     1.007825032230
         O           -0.661515257712     0.184760206605     0.000000000000    15.994914619570
         C            0.669963865101    -0.247863585520     0.000000000000    12.000000000000
         H            1.219446673372     0.109792966232    -0.883261249237     1.007825032230
         H            1.219446673372     0.109792966232     0.883261249237     1.007825032230
         H            0.744204134226    -1.345355445371     0.000000000000     1.007825032230



In [20]:

basename = "butane_scan" # Base name for log files and output CSV file. This will be combined with the 'runname' variable to create unique file names for each run.
runname = "hf-631G" # You can change this to keep track of different runs with different settings. This will be used in the log file names and the output CSV file name.

method = "hf/6-31G*" # You can change this to use a different level of theory and basis set. For example, "mp2/6-31G*", "b3lyp/6-31G*", etc.

start_angle = 0 # Starting dihedral angle in degrees
end_angle = 180 # Ending dihedral angle in degrees
step_size = 10 # Step size for dihedral angle in degrees


# Define base butane geometry (Z-matrix for easy dihedral manipulation)
butane_template = """
0 1
C
C 1 1.54
C 2 1.54 1 112.0
C 3 1.54 2 112.0 1 {phi}
H 1 1.09 2 109.5 3 181.0
H 1 1.09 2 109.5 3 61.0
H 1 1.09 2 109.5 3 -61.0
H 2 1.09 3 109.5 1 181.0
H 2 1.09 3 109.5 1 61.0
H 3 1.09 4 109.5 2 181.0
H 3 1.09 4 109.5 2 61.0
H 4 1.09 3 109.5 2 -61.0
H 4 1.09 3 109.5 2 61.0
H 4 1.09 3 109.5 2 181.0
"""
# I added 1 to all torsion angles to avoid symmetry locks in the structure. This is a common trick to break 
#    symmetry and allow the optimization to proceed without getting stuck in a high-symmetry point.

phi = 80.0

In [21]:
import psi4
import numpy as np
import pandas as pd
import sys
from contextlib import redirect_stdout
from contextlib import redirect_stderr
import time

# Set Psi4 options
psi4.core.set_output_file(f"{basename}_{runname}.log", False) # True: append logs; False: overwrite logs
psi4.set_memory('2 GB')
psi4.set_num_threads(4) # Adjust based on your CPU cores; 1 for single-threaded, more for parallel
psi4.set_options({
    'print': 1,         # 0 for no output, 1 for minimal output, 2 for detailed output. Needs to be set to 1 or greater for optking to work properly.
    'reference': 'rhf', # 'uhf', 'rhf', 'rohf'
    'scf_type': 'df',   # density fitting for faster SCF. Other options are 'pk' (default), 'direct', 'outofcore' and more
    'optking__frozen_dihedral': ''
    })

# define message for header of each optimization run in the loop. This will be printed to the stderr file for each angle, so we can easily see where each optimization starts in the log.
dihedral_print_message = """

--------------------------------------------------
Starting optimization for dihedral angle: {phi:.1f} degrees
--------------------------------------------------

"""

# Define scan range (0 to 360 degrees)
angles = np.arange(start_angle, end_angle + step_size, step_size) # Generate array of dihedral angles from start to end with the specified step size
energies = []
molecules = []
wavefunctions = []

sys.stderr.write(dihedral_print_message.format(phi=0)) # Print to a file for stderr. verbose output from Psi4 will be written to this file.

print(f"Calculating for dihedral = {phi:.1f} deg")
# Create molecule with current dihedral
mol = psi4.geometry(butane_template.format(phi=phi))
# Compute energy (with structural optimization)
start = time.time()

dihedral_string = "1 2 3 4 90.0 90.1"
psi4.set_options({"optking__ranged_dihedral": f"{dihedral_string}"})

###############################
# This is where Psi4 is used. This will use the structure established from 'butane_template' with the 
#     angle 'phi' set to a value from the list 'angles'. The 'optimize' function will perform a geometry 
#     optimization at the specified level of theory (SCF in this case) while keeping the specified 
#     dihedral angle frozen. 'hf' is the level of theory (Hartree-Fock). You can change this to 'mp2', 
#     'ccsd', 'b3lyp', etc. for more accurate calculations, but it will take longer.
###############################
energy, wfn = psi4.optimize(method, molecule=mol, return_wfn=True)
###############################
# End of Psi4 calculation. Everything else is just setup and timing. 

###############################
energies.append(energy) # The final optimized energy from the calculation is stored in the 'energies' list.
wavefunctions.append(wfn) # The wavefunction from the calculation is stored in the 'wavefunctions' list.
end = time.time()
print(f"Time for dihedral optimization {phi:.1f} deg: {end - start:.2f} seconds")
mol = psi4.core.get_active_molecule() # After the optimization, we retrieve the optimized molecule using 'get_active_molecule()'. This allows us to store the optimized geometry for each angle in the 'molecules' list. We do this after the optimization so that we have the final optimized structure for each dihedral angle, rather than just the initial structure that we started with for each angle.            
molecules.append(mol)




--------------------------------------------------
Starting optimization for dihedral angle: 0.0 degrees
--------------------------------------------------



Calculating for dihedral = 80.0 deg


	Change in internal coordinate of 5.96e-01 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 1.
	Change in internal coordinate of 6.57e-01 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 2.
	Change in internal coordinate of 9.18e-01 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 1.
	Change in internal coordinate of 5.85e-01 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 3.
	Change in internal coordinate of 9.20e-01 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 2.
	Change in internal coordinate of 1.18e+00 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 1.
	Change in internal coordinate of 7.77e-01 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 3.
	Change in internal coordinate of 1.11e+00 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 2.
	Change in internal coordinate of 1.37e+00 exceeds limit of 5.00e-01.
	Skipping Hessian update for step 1.
	Previous geometry is closer to targe

Optimizer: Optimization complete!
Time for dihedral optimization 80.0 deg: 26.62 seconds
